In [1]:
pip install transformers torch sentencepiece

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG = "eng_Latn"
TGT_LANG = "luo_Latn"
INPUT_FILE = "/content/drive/MyDrive/Combined_PSA_Raw.csv"
OUTPUT_FILE = "PSA_Translated_NLLB.csv"
BATCH_SIZE = 32
MAX_LENGTH = 512

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=SRC_LANG)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
if device == "cuda":
    model = model.half()  # fp16 on T4 — halves memory, speeds up generation
model.eval()

def translate_batch(texts):
    inputs = tokenizer(texts, return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_LENGTH).to(device)
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(TGT_LANG),
            max_length=MAX_LENGTH,
        )
    return tokenizer.batch_decode(generated, skip_special_tokens=True)

# Drop rows with nothing worth translating, same logic as the cleaning pipeline
df = pd.read_csv(INPUT_FILE)
excel_errors = df.apply(lambda r: r.astype(str).str.contains(r'#VALUE!|#REF!|#N/A', regex=True).any(), axis=1)
df = df[~excel_errors].copy()
df = df.dropna(subset=['English']).copy()
df = df.drop_duplicates(subset=['English'], keep='first').reset_index(drop=True)

print(f"Translating {len(df)} rows (dropped corrupted/duplicate rows first)")

texts = df['English'].astype(str).tolist()
translations = []
for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i + BATCH_SIZE]
    translations.extend(translate_batch(batch))
    done = i + len(batch)
    if done % (BATCH_SIZE * 20) == 0 or done == len(texts):
        print(f"{done}/{len(texts)} done")

df['Dholuo'] = translations
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Saved {OUTPUT_FILE} — {df.shape}")

Using device: cuda


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Translating 9539 rows (dropped corrupted/duplicate rows first)
640/9539 done
1280/9539 done
1920/9539 done
2560/9539 done
3200/9539 done
3840/9539 done
4480/9539 done
5120/9539 done
5760/9539 done
6400/9539 done
7040/9539 done
7680/9539 done
8320/9539 done
8960/9539 done
9539/9539 done
Saved PSA_Translated_NLLB.csv — (9539, 5)
